# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Metadata object

# Print dataset name and description
print(f"{getattr(meta, 'name', '')}: {getattr(meta, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id's
record_sets = dataset.record_sets
print('Available record sets:')
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name')}")

# For this notebook, we select the main record set by inspecting which record set contains the core data
if record_sets:
    # Select the first record set as the primary one
    main_record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set @{main_record_set_id}:")
    for f in record_sets[0].get('field', []):
        print(f"@id: {f['@id']}, name: {f.get('name')} - dtype: {f.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Build a list of record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Columns in primary record set @{main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field to analyze. We'll try to choose 'cr:coverage' or fallback to a numeric column.
df = dataframes[main_record_set_id]
# Try to find a likely numeric field by name
numeric_field = None
for candidate in ['cr:coverage', 'cr:peptideCount', 'cr:mw', 'cr:normalizedAbundance', 'cr:pI']:
    if candidate in df.columns:
        numeric_field = candidate
        break
if not numeric_field:
    # Fallback: First numeric dtype column
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break
if numeric_field is None:
    raise RuntimeError('No numeric field found in the main record set to analyze.')

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy() if numeric_field in filtered_df.columns else []].head())

# Choose a group field to aggregate by, e.g., 'cr:accession' or 'cr:description'
group_field = None
for candidate in ['cr:description', 'cr:accession', 'cr:sample']:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field} in main record set")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field, if available
if group_field:
    top_groups = df[group_field].value_counts().index[:5]
    subset = df[df[group_field].isin(top_groups)]
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=subset)
    plt.title(f"{numeric_field} by {group_field} (top 5 groups)")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains protein mass spectrometry results from stimulated human mast cells, including fields for accession, description, experimental parameters, and abundance measurements.
- We successfully loaded and explored the core data using `mlcroissant`, referencing all entities by their `@id` as per Croissant standards.
- Initial EDA has shown clear variability in measurements such as coverage, with opportunities for more in-depth proteomics analysis, such as identifying outliers, patterns in protein abundances, and grouping by protein accession or description.
- The notebook provides a launchpad for advanced analysis workflows, including statistical testing, clustering proteins, and visualizing group-based effects.
<br>For additional analysis and provenance, consult the dataset's Croissant schema and documentation.